# Demographic Bias Analysis Tool - Google Colab

**Complete functional tool for analyzing demographic bias in facial emotion datasets.**

## Features:
- Upload and analyze emotion datasets
- Detect skin tone bias patterns
- Generate interactive visualizations
- Export results for academic use

**Author:** MIT Capstone Project - Ethical AI Research

## 🔧 Setup and Installation

In [ ]:
# Install required packages
!pip install opencv-python-headless matplotlib seaborn pandas numpy

# Import libraries
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
from collections import defaultdict
import json
import zipfile
from google.colab import files
import warnings

warnings.filterwarnings('ignore')
print("✅ Setup complete!")

## 📁 Upload Dataset

In [ ]:
# Upload your dataset ZIP file
print("📤 Upload your emotion dataset (ZIP file):")
uploaded = files.upload()

# Extract the uploaded file
dataset_path = "/content/dataset"
os.makedirs(dataset_path, exist_ok=True)

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        print(f"📦 Extracting {filename}...")
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall(dataset_path)
        print(f"✅ Extracted to {dataset_path}")
        break

# List contents
print("\n📋 Dataset contents:")
for item in os.listdir(dataset_path):
    item_path = os.path.join(dataset_path, item)
    if os.path.isdir(item_path):
        count = len([f for f in os.listdir(item_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f"  📂 {item}/ ({count} images)")

## 🔍 Bias Analyzer Class

In [ ]:
class BiasAnalyzer:
    def __init__(self, dataset_path):
        self.dataset_path = Path(dataset_path)
        self.results = []
        
    def analyze_image(self, image_path):
        """Analyze single image for skin tone."""
        try:
            img = cv2.imread(str(image_path))
            if img is None:
                return None
                
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            avg_luminance = np.mean(gray)
            
            # Face detection
            try:
                face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
                faces = face_cascade.detectMultiScale(gray, 1.1, 4)
                
                if len(faces) > 0:
                    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
                    face_region = gray[y:y+h, x:x+w]
                    avg_luminance = np.mean(face_region)
                    face_detected = True
                else:
                    face_detected = False
            except:
                face_detected = False
            
            # Categorize skin tone
            if avg_luminance < 80:
                skin_category = 'darker'
            elif avg_luminance < 120:
                skin_category = 'medium'
            else:
                skin_category = 'lighter'
            
            return {
                'luminance': avg_luminance,
                'skin_category': skin_category,
                'face_detected': face_detected
            }
        except:
            return None
    
    def analyze_dataset(self):
        """Analyze entire dataset."""
        print("🔍 Analyzing dataset...")
        
        emotions = ['happy', 'sad', 'angry', 'fear', 'surprise', 'neutral', 'disgust']
        total_processed = 0
        
        for emotion in emotions:
            emotion_path = self.dataset_path / emotion
            if emotion_path.exists():
                print(f"📂 Processing {emotion}...")
                
                image_files = []
                for ext in ['.jpg', '.jpeg', '.png']:
                    image_files.extend(emotion_path.glob(f'*{ext}'))
                    image_files.extend(emotion_path.glob(f'*{ext.upper()}'))
                
                for img_path in image_files:
                    result = self.analyze_image(img_path)
                    if result:
                        result.update({
                            'emotion': emotion,
                            'filename': img_path.name
                        })
                        self.results.append(result)
                        total_processed += 1
        
        self.df = pd.DataFrame(self.results)
        print(f"✅ Analyzed {total_processed} images")
        return self.df
    
    def detect_bias(self):
        """Detect bias patterns."""
        if len(self.df) == 0:
            return {}
        
        skin_dist = self.df['skin_category'].value_counts(normalize=True) * 100
        emotion_dist = self.df['emotion'].value_counts(normalize=True) * 100
        
        bias_indicators = []
        
        # Check underrepresentation
        darker_pct = skin_dist.get('darker', 0)
        if darker_pct < 20:
            bias_indicators.append(f"Darker skin underrepresented: {darker_pct:.1f}%")
        
        # Check emotion bias
        if 'darker' in skin_dist.index:
            darker_data = self.df[self.df['skin_category'] == 'darker']
            darker_emotions = darker_data['emotion'].value_counts(normalize=True) * 100
            
            negative_emotions = ['sad', 'angry', 'fear']
            negative_pct = sum(darker_emotions.get(emo, 0) for emo in negative_emotions)
            overall_negative = sum(emotion_dist.get(emo, 0) for emo in negative_emotions)
            
            if negative_pct > overall_negative + 15:
                bias_indicators.append(f"Emotion bias: {negative_pct:.1f}% negative in darker vs {overall_negative:.1f}% overall")
        
        return {
            'skin_distribution': skin_dist.to_dict(),
            'emotion_distribution': emotion_dist.to_dict(),
            'bias_indicators': bias_indicators,
            'total_samples': len(self.df),
            'face_detection_rate': self.df['face_detected'].mean() * 100
        }
    
    def create_visualizations(self):
        """Create visualizations."""
        if len(self.df) == 0:
            return
        
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        fig.suptitle('Demographic Bias Analysis Results', fontsize=16)
        
        # Skin tone distribution
        skin_counts = self.df['skin_category'].value_counts()
        axes[0, 0].pie(skin_counts.values, labels=skin_counts.index, autopct='%1.1f%%')
        axes[0, 0].set_title('Skin Tone Distribution')
        
        # Emotion distribution
        emotion_counts = self.df['emotion'].value_counts()
        axes[0, 1].bar(emotion_counts.index, emotion_counts.values)
        axes[0, 1].set_title('Emotion Distribution')
        axes[0, 1].tick_params(axis='x', rotation=45)
        
        # Cross-tabulation
        if len(self.df['skin_category'].unique()) > 1:
            crosstab = pd.crosstab(self.df['emotion'], self.df['skin_category'], normalize='columns') * 100
            sns.heatmap(crosstab, annot=True, fmt='.1f', ax=axes[1, 0], cmap='YlOrRd')
            axes[1, 0].set_title('Emotion % by Skin Tone')
        
        # Luminance distribution
        axes[1, 1].hist(self.df['luminance'], bins=20, alpha=0.7)
        axes[1, 1].set_title('Luminance Distribution')
        axes[1, 1].set_xlabel('Average Luminance')
        
        plt.tight_layout()
        plt.show()

print("✅ BiasAnalyzer class ready!")

## 🚀 Run Analysis

In [ ]:
# Initialize analyzer
analyzer = BiasAnalyzer(dataset_path)

# Run analysis
df = analyzer.analyze_dataset()

# Display basic info
if len(df) > 0:
    print(f"\n📊 Dataset Summary:")
    print(f"Total images: {len(df)}")
    print(f"\nSkin tone distribution:")
    skin_dist = df['skin_category'].value_counts(normalize=True) * 100
    for category, pct in skin_dist.items():
        print(f"  {category}: {pct:.1f}%")
    
    print(f"\nEmotion distribution:")
    emotion_dist = df['emotion'].value_counts(normalize=True) * 100
    for emotion, pct in emotion_dist.items():
        print(f"  {emotion}: {pct:.1f}%")
else:
    print("❌ No data found")

## ⚖️ Bias Detection

In [ ]:
# Detect bias patterns
bias_results = analyzer.detect_bias()

print("📈 Bias Analysis Results:")
print("=" * 40)

if bias_results.get('bias_indicators'):
    print("\n⚠️ Bias indicators detected:")
    for indicator in bias_results['bias_indicators']:
        print(f"  • {indicator}")
else:
    print("\n✅ No major bias patterns detected")

print(f"\n👤 Face detection success: {bias_results.get('face_detection_rate', 0):.1f}%")

# Save results
with open('/content/bias_analysis_results.json', 'w') as f:
    json.dump(bias_results, f, indent=2)
print("\n💾 Results saved to bias_analysis_results.json")

## 📊 Generate Visualizations

In [ ]:
# Create visualizations
analyzer.create_visualizations()

## 📥 Download Results

In [ ]:
# Download analysis results
print("📥 Downloading analysis results...")
files.download('/content/bias_analysis_results.json')
print("✅ Download complete!")

## 📋 Summary

This tool analyzes facial emotion datasets for demographic bias by:

1. **Skin Tone Analysis**: Uses luminance-based categorization
2. **Bias Detection**: Identifies underrepresentation and emotion distribution bias
3. **Visualizations**: Creates comprehensive charts and heatmaps
4. **Academic Output**: Generates results suitable for research papers

**For MIT Capstone Integration:**
- Use bias indicators to configure DST uncertainty thresholds
- Implement ethical override systems for biased predictions
- Extract representative samples for peer verification
- Monitor model fairness in production deployment